# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a guide for loading and exploring the [FAIRˆ² mass spectrometry protein dataset](https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json) using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure the `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(url)
metadata = dataset.metadata

print(f"Dataset Title: {getattr(metadata, 'name', '')}\n")
print(f"Description: {getattr(metadata, 'description', '')}\n")
print(f"Keywords: {getattr(metadata, 'keywords', [])}")

## 2. Data Overview
Review available record sets, fields, their `@id`s, and column structures.

In [ ]:
# List the available record sets and their fields
record_sets = []
record_sets_info = []

if hasattr(metadata, 'record_set') and metadata.record_set:
    for rs in metadata.record_set:
        rs_id = getattr(rs, '@id', None)
        rs_name = getattr(rs, 'name', None)
        record_sets.append(rs_id)
        print(f"Record set: {rs_name} (@id={rs_id})")

        # List out the fields for this record set
        if hasattr(rs, 'field'):
            for field in rs.field:
                field_id = getattr(field, '@id', None)
                field_name = getattr(field, 'name', None)
                print(f"    Field: {field_name} (@id={field_id})")
                # If the field is from a column, show the originating column id
                if hasattr(field, 'column'):
                    col = field.column
                    col_id = getattr(col, '@id', None)
                    print(f"        Column: (@id={col_id})")
        print("")
        record_sets_info.append({'id': rs_id, 'name': rs_name})
else:
    print("No record sets found in this dataset.")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s identified above.

In [ ]:
# If there are record sets, extract their data
dataframes = {}
if record_sets:
    for rs_id in record_sets:
        try:
            print(f"Loading records for {rs_id} ...")
            records = list(dataset.records(record_set=rs_id))
            dataframes[rs_id] = pd.DataFrame(records)
            print(f"Loaded {len(records)} rows for record set {rs_id}")
            print(f"Sample columns: {dataframes[rs_id].columns.tolist()[:5]}")
        except Exception as e:
            print(f"Could not load record set {rs_id}: {e}")

    # Use first record set for demonstration if available
    main_record_set = record_sets[0]
    print(f"\nFirst few records from record set {main_record_set}:")
    display(dataframes[main_record_set].head())
else:
    print("No record sets to extract data from.")

## 4. Exploratory Data Analysis (EDA)
Apply data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and grouping data by key attributes.

Replace the variable values to match the specific record set and field `@id`s you wish to analyze.

In [ ]:
# Example analysis for the main record set, adjust field IDs as needed
import numpy as np

# Identify available numeric columns
df = dataframes[main_record_set]
numeric_fields = df.select_dtypes(include=[np.number]).columns.tolist()
print(f"Numeric fields: {numeric_fields}")

if numeric_fields:
    numeric_field = numeric_fields[0]  # Pick first numeric field for demonstration
    threshold = df[numeric_field].mean()  # Use mean as a threshold example

    filtered_df = df[df[numeric_field] > threshold]
    print(f"Filtered records with {numeric_field} > {threshold:.2f}:")
    print(filtered_df.head())

    norm_col = f"{numeric_field}_normalized"
    filtered_df[norm_col] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
    print(f"\nNormalized {numeric_field} for filtered records:")
    print(filtered_df[[numeric_field, norm_col]].head())

    # Try to find a categorical column to group by
    possible_groups = df.select_dtypes(include=[object]).columns.tolist()
    group_field = None
    for candidate in possible_groups:
        if df[candidate].nunique() < min(20, len(df)):  # choose a reasonable groupby key
            group_field = candidate
            break

    if group_field:
        grouped_df = filtered_df.groupby(group_field, as_index=False)[numeric_field].mean()
        print(f"\nGrouped data by {group_field}, averaging {numeric_field}:")
        print(grouped_df.head())
else:
    print("No numeric fields available for analysis.")

## 5. Visualization
Visualize data distributions for numeric and categorical fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if numeric_fields:
    plt.figure(figsize=(8, 5))
    sns.histplot(df[numeric_field], kde=True, bins=30)
    plt.title(f"Distribution of {numeric_field}")
    plt.xlabel(numeric_field)
    plt.ylabel("Frequency")
    plt.show()

    if group_field:
        plt.figure(figsize=(10, 5))
        sns.boxplot(x=group_field, y=numeric_field, data=df)
        plt.title(f"{numeric_field} by {group_field}")
        plt.xlabel(group_field)
        plt.ylabel(numeric_field)
        plt.xticks(rotation=45)
        plt.show()

## 6. Conclusion
This notebook demonstrated how to load, explore, and perform preliminary analysis on the FAIRˆ² mass spectrometry dataset using `mlcroissant`. 

We:  
- Loaded standardized dataset metadata and explored its structure.  
- Reviewed available record sets and fields (by their `@id`).  
- Loaded records as DataFrames for processing.  
- Filtered, normalized, and visualized key numeric and categorical variables.  

**Next steps:** Analyze domain-specific patterns (e.g., protein abundance trends), enrich analysis with additional domain knowledge, or integrate with downstream ML workflows.